In [1]:
import csv
import os
import hashlib
import uuid
from datetime import datetime

USERS_FILE = "users.csv"
RIDES_FILE = "rides.csv"
BOOKINGS_FILE = "bookings.csv"

USER_FIELDS = ["user_id", "name", "student_id", "email", "password_hash"]

RIDE_FIELDS = [
    "ride_id", "driver_id", "origin", "destination",
    "travel_date", "departure_time", "seats_total",
    "seats_available", "cost_per_seat", "notes", "status"
]

BOOKING_FIELDS = [
    "booking_id", "ride_id", "passenger_id",
    "seats_booked", "booking_time", "status"
]


def initialize_files():
    """Create CSV files if they do not exist."""

    files = {
        USERS_FILE: USER_FIELDS,
        RIDES_FILE: RIDE_FIELDS,
        BOOKINGS_FILE: BOOKING_FIELDS
    }

    for filename, headers in files.items():
        try:
            if not os.path.exists(filename):
                with open(filename, "w", newline="", encoding="utf-8") as file:
                    writer = csv.DictWriter(file, fieldnames=headers)
                    writer.writeheader()

        except PermissionError:
            print("Permission denied:", filename)

        except OSError as error:
            print("File error:", error)


def read_records(filename):
    """Read records from CSV file."""

    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            return list(csv.DictReader(file))

    except FileNotFoundError:
        initialize_files()
        return []

    except PermissionError:
        print("Permission denied:", filename)
        return []


def append_record(filename, fields, record):
    """Save a new record in CSV file."""

    try:
        with open(filename, "a", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fields)
            writer.writerow(record)

        return True

    except (PermissionError, OSError, csv.Error) as error:
        print("Error saving record:", error)
        return False


def update_csv_file(filename, fields, records):
    """Update all records in a CSV file."""

    try:
        with open(filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fields)
            writer.writeheader()
            writer.writerows(records)

        return True

    except (PermissionError, OSError, csv.Error) as error:
        print("Error updating file:", error)
        return False


initialize_files()

print("Campus Carpool Portal Initialized Successfully!")

Campus Carpool Portal Initialized Successfully!


In [2]:
def hash_password(password):
    """Convert password into a secure hash."""

    return hashlib.sha256(password.encode()).hexdigest()


def register_user():
    """Register a new student."""

    print("\n===== STUDENT REGISTRATION =====")

    name = input("Enter your full name: ")
    student_id = input("Enter your student ID: ")
    email = input("Enter your university email: ").lower()
    password = input("Create your password: ")

    if not all([name.strip(), student_id.strip(), email.strip(), password]):
        print("All fields are required!")
        return

    users = read_records(USERS_FILE)

    for user in users:

        if user["student_id"].lower() == student_id.lower():
            print("Student ID already registered!")
            return

        if user["email"].lower() == email:
            print("Email already registered!")
            return

    new_user = {
        "user_id": str(uuid.uuid4())[:8],
        "name": name,
        "student_id": student_id,
        "email": email,
        "password_hash": hash_password(password)
    }

    if append_record(USERS_FILE, USER_FIELDS, new_user):
        print("\nRegistration Successful!")
        print("Your User ID:", new_user["user_id"])


def user_login():
    """Login an existing student."""

    print("\n===== STUDENT LOGIN =====")

    email = input("Enter email: ").lower()
    password = input("Enter password: ")

    password_hash = hash_password(password)

    users = read_records(USERS_FILE)

    for user in users:

        if (user["email"].lower() == email and
                user["password_hash"] == password_hash):

            print("\nLogin Successful!")
            print("Welcome", user["name"])

            return user

    print("Invalid email or password!")

    return None

In [3]:
def add_record(driver):
    """Allow a student to offer a ride."""

    print("\n===== OFFER A RIDE =====")

    origin = input("Enter starting location: ")
    destination = input("Enter destination: ")

    travel_date = input("Enter travel date (YYYY-MM-DD): ")
    departure_time = input("Enter departure time (HH:MM): ")

    try:
        date_obj = datetime.strptime(travel_date, "%Y-%m-%d").date()
        time_obj = datetime.strptime(departure_time, "%H:%M").time()

        if date_obj < datetime.now().date():
            print("Travel date cannot be in the past!")
            return

        seats = int(input("Enter available seats: "))
        cost = float(input("Enter cost per seat: "))

        if seats <= 0 or cost < 0:
            print("Invalid seats or cost!")
            return

    except ValueError:
        print("Invalid date, time, or numeric value!")
        return

    notes = input("Additional details: ")

    ride = {
        "ride_id": str(uuid.uuid4())[:8],
        "driver_id": driver["user_id"],
        "origin": origin,
        "destination": destination,
        "travel_date": date_obj.isoformat(),
        "departure_time": time_obj.strftime("%H:%M"),
        "seats_total": str(seats),
        "seats_available": str(seats),
        "cost_per_seat": str(cost),
        "notes": notes,
        "status": "active"
    }

    if append_record(RIDES_FILE, RIDE_FIELDS, ride):

        print("\nRide Added Successfully!")
        print("Your Ride ID:", ride["ride_id"])

In [4]:
def search_record():
    """Search rides according to location."""

    print("\n===== SEARCH RIDES =====")

    origin = input("Enter starting location: ").lower()
    destination = input("Enter destination: ").lower()

    rides = read_records(RIDES_FILE)

    found = False

    for ride in rides:

        if ride["status"] != "active":
            continue

        if int(ride["seats_available"]) <= 0:
            continue

        if (origin in ride["origin"].lower() and
                destination in ride["destination"].lower()):

            found = True

            driver = None

            for user in read_records(USERS_FILE):
                if user["user_id"] == ride["driver_id"]:
                    driver = user
                    break

            driver_name = driver["name"] if driver else "Unknown"

            print("\n----------------------------")
            print("Ride ID:", ride["ride_id"])
            print("Driver:", driver_name)
            print("From:", ride["origin"])
            print("To:", ride["destination"])
            print("Date:", ride["travel_date"])
            print("Time:", ride["departure_time"])
            print("Available Seats:", ride["seats_available"])
            print("Cost Per Seat:", ride["cost_per_seat"])
            print("Details:", ride["notes"])
            print("----------------------------")

    if not found:
        print("No available rides found!")

In [5]:
def book_ride(passenger):
    """Allow students to book available seats."""

    print("\n===== BOOK A RIDE =====")

    search_record()

    ride_id = input("\nEnter Ride ID to book: ")

    rides = read_records(RIDES_FILE)
    bookings = read_records(BOOKINGS_FILE)

    selected_ride = None

    for ride in rides:

        if ride["ride_id"] == ride_id and ride["status"] == "active":
            selected_ride = ride
            break

    if selected_ride is None:
        print("Ride not found!")
        return

    if selected_ride["driver_id"] == passenger["user_id"]:
        print("You cannot book your own ride!")
        return

    for booking in bookings:

        if (booking["ride_id"] == ride_id and
                booking["passenger_id"] == passenger["user_id"] and
                booking["status"] == "active"):

            print("You already booked this ride!")
            return

    try:
        seats = int(input("Enter number of seats: "))

        available = int(selected_ride["seats_available"])

        if seats <= 0 or seats > available:
            print("Invalid number of seats!")
            return

    except ValueError:
        print("Please enter a valid number!")
        return

    selected_ride["seats_available"] = str(available - seats)

    booking = {
        "booking_id": str(uuid.uuid4())[:8],
        "ride_id": ride_id,
        "passenger_id": passenger["user_id"],
        "seats_booked": str(seats),
        "booking_time": datetime.now().isoformat(timespec="seconds"),
        "status": "active"
    }

    if update_csv_file(RIDES_FILE, RIDE_FIELDS, rides):

        if append_record(BOOKINGS_FILE, BOOKING_FIELDS, booking):
            print("\nRide Booked Successfully!")
            print("Booking ID:", booking["booking_id"])

        else:
            selected_ride["seats_available"] = str(available)
            update_csv_file(RIDES_FILE, RIDE_FIELDS, rides)

    else:
        print("Booking unsuccessful!")

In [6]:
def view_my_bookings(user):
    """Display bookings of the logged-in student."""

    print("\n===== MY BOOKINGS =====")

    bookings = read_records(BOOKINGS_FILE)
    rides = read_records(RIDES_FILE)

    found = False

    for booking in bookings:

        if booking["passenger_id"] == user["user_id"]:

            found = True

            ride = None

            for r in rides:
                if r["ride_id"] == booking["ride_id"]:
                    ride = r
                    break

            print("\nBooking ID:", booking["booking_id"])
            print("Ride ID:", booking["ride_id"])

            if ride:
                print("Route:", ride["origin"], "to", ride["destination"])

            print("Seats:", booking["seats_booked"])
            print("Status:", booking["status"])

    if not found:
        print("No bookings found!")

In [7]:
def cancel_booking(user):
    """Cancel an active booking and return seats to the ride."""

    print("\n===== CANCEL BOOKING =====")

    booking_id = input("Enter Booking ID: ")

    bookings = read_records(BOOKINGS_FILE)
    rides = read_records(RIDES_FILE)

    target = None

    for booking in bookings:

        if (booking["booking_id"] == booking_id and
                booking["passenger_id"] == user["user_id"] and
                booking["status"] == "active"):

            target = booking
            break

    if target is None:
        print("Active booking not found!")
        return

    selected_ride = None

    for ride in rides:

        if ride["ride_id"] == target["ride_id"]:
            selected_ride = ride
            break

    if selected_ride is None:
        print("Associated ride not found!")
        return

    try:
        selected_ride["seats_available"] = str(
            int(selected_ride["seats_available"]) +
            int(target["seats_booked"])
        )

    except ValueError:
        print("Invalid stored seat information!")
        return

    target["status"] = "cancelled"

    if update_csv_file(RIDES_FILE, RIDE_FIELDS, rides):

        if update_csv_file(BOOKINGS_FILE, BOOKING_FIELDS, bookings):
            print("Booking cancelled successfully!")
        else:
            print("Could not update booking records.")

    else:
        print("Could not update ride records.")

In [8]:
def view_my_rides(driver):
    """Display rides offered by the logged-in driver."""

    print("\n===== MY OFFERED RIDES =====")

    rides = read_records(RIDES_FILE)

    found = False

    for ride in rides:

        if ride["driver_id"] == driver["user_id"]:

            found = True

            print("\nRide ID:", ride["ride_id"])
            print("From:", ride["origin"])
            print("To:", ride["destination"])
            print("Date:", ride["travel_date"])
            print("Time:", ride["departure_time"])
            print("Available Seats:", ride["seats_available"])
            print("Status:", ride["status"])

    if not found:
        print("You have not offered any rides.")

In [9]:
def generate_report():
    """Generate a summary report of portal activities."""

    users = read_records(USERS_FILE)
    rides = read_records(RIDES_FILE)
    bookings = read_records(BOOKINGS_FILE)

    active_rides = 0
    available_seats = 0
    active_bookings = 0

    for ride in rides:

        if ride["status"] == "active":

            active_rides += 1

            try:
                available_seats += int(ride["seats_available"])
            except ValueError:
                pass

    for booking in bookings:

        if booking["status"] == "active":
            active_bookings += 1

    print("\n====================================")
    print("      CAMPUS CARPOOL REPORT")
    print("====================================")

    print("Registered Students:", len(users))
    print("Total Ride Offers:", len(rides))
    print("Active Rides:", active_rides)
    print("Available Seats:", available_seats)
    print("Total Bookings:", len(bookings))
    print("Active Bookings:", active_bookings)

    print("====================================")

In [10]:
def student_session(user):
    """Display the student dashboard."""

    while True:

        print("\n========== STUDENT DASHBOARD ==========")

        print("Welcome,", user["name"])

        print("1. Offer a Ride")
        print("2. Search Rides")
        print("3. Book a Ride")
        print("4. View My Bookings")
        print("5. Cancel Booking")
        print("6. View My Offered Rides")
        print("7. Logout")

        choice = input("\nEnter your choice: ")

        if choice == "1":
            add_record(user)

        elif choice == "2":
            search_record()

        elif choice == "3":
            book_ride(user)

        elif choice == "4":
            view_my_bookings(user)

        elif choice == "5":
            cancel_booking(user)

        elif choice == "6":
            view_my_rides(user)

        elif choice == "7":
            print("Logged out successfully!")
            break

        else:
            print("Invalid choice! Try again.")

In [11]:
def main():
    """Main function of Campus Carpool Portal."""

    initialize_files()

    print("\n")
    print("==========================================")
    print("   CAMPUS CARPOOL & RIDE-SHARING PORTAL")
    print("==========================================")

    while True:

        print("\n========== MAIN MENU ==========")

        print("1. Student Registration")
        print("2. Student Login")
        print("3. Generate Portal Report")
        print("4. Exit")

        choice = input("\nEnter your choice: ")

        try:

            if choice == "1":
                register_user()

            elif choice == "2":

                user = user_login()

                if user is not None:
                    student_session(user)

            elif choice == "3":
                generate_report()

            elif choice == "4":

                print("\nThank you for using Campus Carpool Portal!")
                break

            else:
                print("Invalid choice! Please select 1-4.")

        except Exception as error:
            print("An unexpected error occurred:", error)

        finally:
            print("\nReturning to main menu...")


# START THE PORTAL
main()



   CAMPUS CARPOOL & RIDE-SHARING PORTAL

========== MAIN MENU ==========
1. Student Registration
2. Student Login
3. Generate Portal Report
4. Exit



Enter your choice:  1



===== STUDENT REGISTRATION =====


Enter your full name:  Kashif Sooomro
Enter your student ID:  i255008@lhr.nu.edu.pk
Enter your university email:  i255008@lhr.nu.edu.pk
Create your password:  qwerty543



Registration Successful!
Your User ID: d26044b6

Returning to main menu...

========== MAIN MENU ==========
1. Student Registration
2. Student Login
3. Generate Portal Report
4. Exit



Enter your choice:  2


... (9 outputs left out)


Enter your choice:  7


Logged out successfully!

Returning to main menu...

========== MAIN MENU ==========
1. Student Registration
2. Student Login
3. Generate Portal Report
4. Exit



Enter your choice:  4



Thank you for using Campus Carpool Portal!

Returning to main menu...
